# 🧠 Notebook 05 — Model Training

## 🎯 Objective
This notebook focuses on training, validating, and evaluating machine learning models
to **predict the likelihood of heart disease** based on clinical features.

It builds on the **preprocessed and feature-engineered data** produced in the previous notebooks:
- `02_data_loading_preprocessing.ipynb` → cleaned and split the dataset.
- `04_feature_engineering.ipynb` → created derived features and saved the preprocessing pipeline (`preprocessor.pkl`).

Here, we integrate that preprocessor into model training pipelines and
experiment with several algorithms to identify the best-performing model.

---

## ⚙️ Workflow Overview
1. **Load processed data** (`X_train`, `X_val`, `X_test`, and corresponding labels).
2. **Load or rebuild the preprocessing pipeline** from `src/features.py`.
3. **Train baseline models**:
   - Logistic Regression
   - Random Forest
   - (Optionally) XGBoost
4. **Evaluate with cross-validation** on the training set (ROC-AUC metric).
5. **Validate performance** on the validation set and save model metrics.
6. **Select the best model** based on validation AUC and Average Precision.
7. **Evaluate the selected model** on the hold-out test set.
8. **Save trained models and metrics** under `/models/` and `/reports/`.
9. (Optional) **Fine-tune hyperparameters** and log results for comparison.

---

## 📦 Outputs
This notebook generates the following artifacts:

| Output File | Description |
|--------------|-------------|
| `models/heart_model_<timestamp>.pkl` | Serialized trained model pipeline |
| `reports/val_metrics_<timestamp>.json` | Validation metrics (AUC, AP) |
| `reports/test_metrics_<timestamp>.json` | Final test performance metrics |
| `reports/best_model.json` | Pointer to the best-performing model |

---

## 🧩 Dependencies
Before running this notebook, ensure that:
- `src/data.py` and `src/features.py` are created (from `03_module_creation.ipynb`).
- The processed datasets exist under `data/processed/`.
- The preprocessing pipeline is saved in `models/preprocessor.pkl` or can be rebuilt using `build_preprocessor()`.

---

## 🚀 Goal
By the end of this notebook, you will have:
- A fully trained and validated heart disease prediction model.
- Quantitative metrics (ROC-AUC, Average Precision, Confusion Matrix).
- A serialized model ready for deployment or interpretability analysis in `06_model_evaluation.ipynb`.

---


## 0) Setup & imports

In [2]:
# (Colab only) Mount and cd to your project root
IN_COLAB = "google.colab" in str(get_ipython())
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/Dr. Taiwo famuyiwa - Data Science & Biostatistics Portfolio/Machine Learning Projects/Heart-Disease-Prediction


Mounted at /content/drive
/content/drive/MyDrive/Dr. Taiwo famuyiwa - Data Science & Biostatistics Portfolio/Machine Learning Projects/Heart-Disease-Prediction


In [12]:
# Ensure we can import from src/
import os, sys, json, time, warnings
warnings.filterwarnings("ignore")
sys.path.append(os.getcwd())

import numpy as np
import pandas as pd
from pathlib import Path

# Sklearn
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    confusion_matrix, classification_report
)
from sklearn.model_selection import StratifiedKFold, cross_val_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Optional: XGBoost (install first if needed)
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

# Project modules
from src.data import PROC_DIR
from src.features import build_preprocessor  # or load saved one if you prefer


# Paths
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True, parents=True)
REPORTS_DIR = Path("reports")
REPORTS_DIR.mkdir(exist_ok=True, parents=True)

In [4]:
X_train = pd.read_csv(PROC_DIR / "X_train.csv")
y_train = pd.read_csv(PROC_DIR / "y_train.csv").squeeze("columns")

X_val   = pd.read_csv(PROC_DIR / "X_val.csv")
y_val   = pd.read_csv(PROC_DIR / "y_val.csv").squeeze("columns")

X_test  = pd.read_csv(PROC_DIR / "X_test.csv")
y_test  = pd.read_csv(PROC_DIR / "y_test.csv").squeeze("columns")

X_train.shape, X_val.shape, X_test.shape

((194, 13), (49, 13), (61, 13))

## 2) Prepare / Load Preprocessor

In [5]:
import joblib
pre = joblib.load("models/preprocessor.pkl")  # fitted

## 3) Define Models

In [6]:
def build_models(random_state=42):
    models = {
        "logreg": LogisticRegression(
            max_iter=200, class_weight="balanced", random_state=random_state
        ),
        "rf": RandomForestClassifier(
            n_estimators=500, max_depth=None, min_samples_leaf=1,
            class_weight="balanced", random_state=random_state, n_jobs=-1
        )
    }
    if HAS_XGB:
        models["xgb"] = XGBClassifier(
            n_estimators=600, max_depth=4, learning_rate=0.05, subsample=0.9,
            colsample_bytree=0.9, reg_lambda=1.0, reg_alpha=0.0,
            objective="binary:logistic", eval_metric="aucpr",
            random_state=random_state, n_jobs=-1
        )
    return models

models = build_models()
list(models.keys())


['logreg', 'rf', 'xgb']

## 4) CV Helper (Train on Train, Score by CV ROC-AUC)

In [7]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict

def cv_score_model(model, X, y, preprocessor, cv_splits=5, seed=42, scoring="roc_auc"):
    pipe = Pipeline([("pre", preprocessor), ("model", model)])
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=seed)
    scores = cross_val_score(pipe, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    return scores.mean(), scores.std()

for name, model in models.items():
    mean_auc, std_auc = cv_score_model(model, X_train, y_train, pre)
    print(f"{name:>6} | CV ROC-AUC: {mean_auc:.3f} ± {std_auc:.3f}")


logreg | CV ROC-AUC: 0.899 ± 0.019
    rf | CV ROC-AUC: 0.890 ± 0.022
   xgb | CV ROC-AUC: 0.869 ± 0.034


## 5) Fit on Train, Evaluate on Validation

In [11]:
def fit_and_validate(model_name, model, X_tr, y_tr, X_va, y_va, preprocessor):
    run_id = f"{model_name}_seed42_{int(time.time())}"
    pipe = Pipeline([("pre", preprocessor), ("model", model)])
    pipe.fit(X_tr, y_tr)

    # Probabilities on validation set
    val_proba = pipe.predict_proba(X_va)[:, 1]
    val_auc   = roc_auc_score(y_va, val_proba)
    val_ap    = average_precision_score(y_va, val_proba)

    # Save artifact + metrics
    import joblib
    model_path = MODELS_DIR / f"heart_model_{run_id}.pkl"
    joblib.dump(pipe, model_path)

    metrics = {"run_id": run_id, "val_auc": float(val_auc), "val_ap": float(val_ap)}
    (REPORTS_DIR / f"val_metrics_{run_id}.json").write_text(json.dumps(metrics, indent=2))

    print(f"💾 Saved model → {model_path}")
    print(f"📝 Val metrics → AUC={val_auc:.3f}  AP={val_ap:.3f}")
    return run_id, metrics

val_results = []
for name, model in models.items():
    run_id, m = fit_and_validate(name, model, X_train, y_train, X_val, y_val, pre)
    val_results.append((name, run_id, m["val_auc"], m["val_ap"]))

pd.DataFrame(val_results, columns=["model","run_id","val_auc","val_ap"]).sort_values("val_auc", ascending=False)


💾 Saved model → models/heart_model_logreg_seed42_1761362412.pkl
📝 Val metrics → AUC=0.967  AP=0.969
💾 Saved model → models/heart_model_rf_seed42_1761362412.pkl
📝 Val metrics → AUC=0.952  AP=0.951
💾 Saved model → models/heart_model_xgb_seed42_1761362413.pkl
📝 Val metrics → AUC=0.931  AP=0.939


,model,run_id,val_auc,val_ap
0,logreg,logreg_seed42_1761362412,0.966555,0.969337
1,rf,rf_seed42_1761362412,0.951505,0.951063
2,xgb,xgb_seed42_1761362413,0.931438,0.939305


## 6) Pick Best by validation AUC, Evaluate on Test

In [13]:
# Choose the best by val AUC
df_val = pd.DataFrame(val_results, columns=["model","run_id","val_auc","val_ap"])
best_row = df_val.sort_values(["val_auc","val_ap"], ascending=False).iloc[0]
best_row

,0
model,logreg
run_id,logreg_seed42_1761362412
val_auc,0.966555
val_ap,0.969337


In [14]:
# Load best and evaluate on TEST
import joblib, json
best_model_path = MODELS_DIR / f"heart_model_{best_row.run_id}.pkl"
best_pipe = joblib.load(best_model_path)

test_proba = best_pipe.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, test_proba)
test_ap  = average_precision_score(y_test, test_proba)

print(f"🏁 TEST → AUC={test_auc:.3f}  AP={test_ap:.3f}")

# Save final test metrics
test_metrics = {
    "best_model": best_row.model,
    "run_id": best_row.run_id,
    "test_auc": float(test_auc),
    "test_ap": float(test_ap)
}
(REPORTS_DIR / f"test_metrics_{best_row.run_id}.json").write_text(json.dumps(test_metrics, indent=2))


🏁 TEST → AUC=0.906  AP=0.877


135

## Threshold & Confusion Matrix on Test

In [15]:
from sklearn.metrics import confusion_matrix, classification_report

thr = 0.5
test_pred = (test_proba >= thr).astype(int)
cm = confusion_matrix(y_test, test_pred)
print(cm)
print(classification_report(y_test, test_pred, digits=3))


[[27  6]
 [ 4 24]]
              precision    recall  f1-score   support

           0      0.871     0.818     0.844        33
           1      0.800     0.857     0.828        28

    accuracy                          0.836        61
   macro avg      0.835     0.838     0.836        61
weighted avg      0.838     0.836     0.836        61



## 8) Simple Hyperparameter Tuning

In [16]:
from sklearn.model_selection import GridSearchCV

grid = {
    "model__C": [0.1, 0.5, 1.0, 2.0],
    "model__penalty": ["l2"],
    "model__solver": ["lbfgs"],
    "model__class_weight": ["balanced", None],
}
pipe = Pipeline([("pre", pre), ("model", LogisticRegression(max_iter=500, random_state=42))])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

gs = GridSearchCV(pipe, grid, cv=cv, scoring="roc_auc", n_jobs=-1, verbose=1)
gs.fit(X_train, y_train)

print("Best params:", gs.best_params_)
print("Best CV AUC:", gs.best_score_)

# Validate + save tuned model
val_proba = gs.predict_proba(X_val)[:, 1]
val_auc   = roc_auc_score(y_val, val_proba)
val_ap    = average_precision_score(y_val, val_proba)
print(f"Tuned Val → AUC={val_auc:.3f} AP={val_ap:.3f}")

import joblib, time
run_id = f"logreg_tuned_seed42_{int(time.time())}"
joblib.dump(gs.best_estimator_, MODELS_DIR / f"heart_model_{run_id}.pkl")
(REPORTS_DIR / f"val_metrics_{run_id}.json").write_text(json.dumps(
    {"run_id": run_id, "val_auc": float(val_auc), "val_ap": float(val_ap)}, indent=2))


Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best params: {'model__C': 0.1, 'model__class_weight': None, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}
Best CV AUC: 0.9056956115779645
Tuned Val → AUC=0.963 AP=0.964


113

## 9) Compare All Runs

In [17]:
def collect_runs():
    rows = []
    for p in REPORTS_DIR.glob("val_metrics_*.json"):
        m = json.loads(p.read_text())
        rows.append({"run_id": m["run_id"], "val_auc": m["val_auc"], "val_ap": m["val_ap"]})
    return pd.DataFrame(rows).sort_values("val_auc", ascending=False)

collect_runs()


,run_id,val_auc,val_ap
0,logreg_seed42_1761362412,0.966555,0.969337
3,logreg_tuned_seed42_1761363485,0.963211,0.964092
1,rf_seed42_1761362412,0.951505,0.951063
2,xgb_seed42_1761362413,0.931438,0.939305


## 10) Save "Best" Alias for Deployment

In [18]:
# Create/overwrite a pointer file with best run_id for downstream scripts
best_pointer = {"best_run_id": best_row.run_id, "model": best_row.model}
(REPORTS_DIR / "best_model.json").write_text(json.dumps(best_pointer, indent=2))
print("🔖 Wrote reports/best_model.json")


🔖 Wrote reports/best_model.json
